# DAT09 · 미션2 파일을 읽고 코드와 단위 및 시각 맞추기 (교수자·해설용 워크북 — 예시 풀이 포함)

- **예상 실습 시간**: 2시간 (전체 10개 미션 총 27시간 과정)
- **작업 폴더**: `DAT09_reference`
- **연결 파일**: 첫 셀에서 `DAT09_3.md`(또는 `DAT09.md`)를 연결하세요.
- **이전 미션 연계**: 같은 작업 폴더에서 미션1~미션1을 먼저 실행했다면, 첫 셀이 `src/m01.py`~`src/m01.py`를 불러와 작업 사전 `c`를 자동으로 복원합니다.

> 학습자 배포 시에는 `TODO(n, step)` 상태인 `DAT09_m02_student.ipynb`만 배포하고, 본 워크북은 해설·채점 참고용으로 분리 보관하세요.


In [ ]:
from pathlib import Path, PurePosixPath
import sys, io, json, re, zipfile, base64, hashlib, importlib
from IPython.display import display, HTML
import html
import pandas as pd
ROLE = 'solution'
EXPECTED_SHA256 = 'a0adfd7b9a2416cd74af2fc99bb7978ad662eec8d9e40ce5b8a539977c2cb17e'
c = None

def connect_md(path):
    global c, ROOT, HERE
    path = Path(path)
    if path.name not in ('DAT09.md', 'DAT09_3.md'):
        raise ValueError('원자료 DAT09.md 또는 DAT09_3.md를 선택하세요. 정답코드 MD는 연결 파일이 아닙니다.')
    text = path.read_text(encoding='utf-8-sig')
    found = re.search(r'<!-- DAT09_RUNTIME_BUNDLE_B64:([A-Za-z0-9+/=]+) -->', text)
    if not found:
        raise ValueError('이 노트북용 원자료 블록이 없습니다. 함께 받은 DAT09.md를 사용하세요.')
    raw = base64.b64decode(found.group(1), validate=True)
    if hashlib.sha256(raw).hexdigest() != EXPECTED_SHA256:
        raise ValueError('자료와 노트북 버전이 다릅니다. 같은 묶음의 파일을 사용하세요.')
    ROOT = Path.cwd() / ('DAT09_reference' if ROLE == 'solution' else 'DAT09_practice')
    ROOT.mkdir(exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(raw)) as archive:
        for info in archive.infolist():
            name = PurePosixPath(info.filename)
            if name.is_absolute() or '..' in name.parts:
                raise ValueError('허용되지 않은 경로입니다.')
            target = ROOT.joinpath(*name.parts)
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(archive.read(info))
    sys.path.insert(0, str(ROOT))
    for name in list(sys.modules):
        if name in ['dat09_common', 'dat09_ui'] or name.startswith('src.m'):
            del sys.modules[name]
    import dat09_common
    globals().update({name: getattr(dat09_common, name) for name in dir(dat09_common) if not name.startswith('_')})
    import collections, itertools, queue, threading, tempfile, importlib.util, shutil
    globals().update({k:v for k,v in locals().items() if k in ['collections','itertools','re','queue','threading','tempfile','importlib','shutil']})
    from dat09_ui import bars, heatmap, line_chart, dashboard_fragment
    globals().update(bars=bars,heatmap=heatmap,line_chart=line_chart,dashboard_fragment=dashboard_fragment)
    c = dat09_common.context(ROOT)
    HERE = ROOT.parent / 'instructor'
    for candidate in [Path.cwd()/'instructor', Path.cwd(), ROOT.parent/'instructor']:
        if (candidate/'aar_key.json').is_file():
            HERE = candidate
            c['instructor_root'] = candidate
            break
    print('자료 연결 완료:', ROOT)
    print('10개 미션·50개 함수·27시간 과정입니다. 미션1부터 실행하세요.')
    print('TODO가 남아 있으면 해당 단계에서 멈춥니다. src에는 작성한 함수를 저장합니다.')
    return c

def hydrate_previous_missions(c, up_to=1):
    """같은 작업 폴더에 저장된 이전 미션(src/m01.py~)을 순차 실행해 작업 사전 c를 복원합니다."""
    import importlib.util
    src_dir = c['root'] / 'src'
    missing = [k for k in range(1, up_to + 1) if not (src_dir / f'm{k:02d}.py').is_file()]
    if missing:
        miss_str = ', '.join(f'미션{k}(DAT09_m{k:02d}_*.ipynb)' for k in missing)
        print(f'[안내] 아직 완료되지 않은 이전 미션이 있습니다: {miss_str}')
        print('이전 미션 노트북을 같은 폴더에서 먼저 실행한 뒤 이 셀을 다시 실행하세요.')
        return False
    for k in range(1, up_to + 1):
        if k in c['completed']:
            continue
        mod_path = src_dir / f'm{k:02d}.py'
        spec = importlib.util.spec_from_file_location(f'dat09_prev_m{k:02d}', mod_path)
        mod = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(mod)
        fns = [getattr(mod, f'm{k:02d}_s{s}') for s in range(1, 6)]
        run_mission(c, k, fns)
    print(f'이전 미션(미션1~미션{up_to}) 상태 복원 완료.')
    return True

DATA_MD = next((Path(p) for p in ('DAT09.md', 'DAT09_3.md') if Path(p).is_file()), None)
if DATA_MD is not None:
    connect_md(DATA_MD)
    hydrate_previous_missions(c, 1)
else:
    print('DAT09_3.md(또는 DAT09.md)를 이 노트북과 같은 폴더에 두고 첫 셀을 다시 실행하세요.')
    print("다른 경로에 있다면 connect_md('/자료가_있는_경로/DAT09_3.md'); hydrate_previous_missions(c)를 실행하세요.")


## 미션2 파일을 읽고 코드와 단위 및 시각 맞추기

예상 2시간입니다.

#### 이번 미션의 목표

세 형식을 실제로 읽어 같은 코드·단위·시각으로 맞추고, 쓸 수 없는 값과 보완 후보를 원자료와 구분해 보관합니다.

### 1단계 세 형식의 파일을 읽고 출처 남기기

#### 이 단계에서 하는 일

화면에 같은 표처럼 보여도 CSV·JSON·TSV는 읽는 방법이 다릅니다. 읽기 과정에서 행을 빠뜨리거나 항목 이름을 잘못 붙이면 뒤의 분석도 틀립니다. 오류가 생겼을 때 원파일로 돌아갈 수 있는 정보도 필요합니다.

#### 시작 전 확인

미션1까지 완료한 결과를 사용합니다. 노트북의 m02_s1(c)를 작성하세요. c는 같은 작업의 자료와 함수를 담는 사전입니다. 필요한 결과를 c에 보관하고 화면에 보여 줄 표를 반환합니다.

#### 사용할 자료

data의 supply.csv, maintenance.json, ammunition.tsv와 config/source_contracts.json의 field_mapping을 사용합니다. 미션1을 먼저 마치고 c['contract']와 세 출처 표본을 준비합니다.

#### 작업 순서

- 출처와 파일 경로를 받아 보고 목록을 반환하는 읽기 함수를 만드세요. CSV·TSV의 구분자를 구별하고 JSON은 records 목록을 읽습니다.

- field_mapping으로 원래 항목 이름을 공통 이름에 연결하세요. row_id, source, item_code, site_code, quantity, unit, as_of, observed_at, received_at, metric, version이 갖춰졌는지 검사합니다.

- 각 보고에 source_file, line_number, raw_payload를 남기세요. line_number는 헤더를 제외한 보고 순번이고 raw_payload는 변환 전 보고 전체입니다. 총행 수와 파일별 해시도 저장합니다.

- 읽기 함수는 c['load_fn'], 전체 보고는 c['raw_rows']에 보관해 다음 단계가 사용할 수 있게 하세요.


#### 저장할 결과

ingest_manifest.json에 파일별 720행과 해시를, raw_rows.csv에 출처를 추적할 수 있는 전체 보고를 저장합니다.

#### 직접 확인하는 방법

총 2,160행의 row_id가 원파일과 대응하는지 확인하세요. JSON의 메타정보를 보고 한 행으로 잘못 세거나 TSV 전체 줄을 한 항목으로 읽지 않았는지도 확인합니다.

#### 실행과 기록

이 미션의 다섯 함수를 작성한 뒤 코드셀을 실행합니다. 반환 표는 outputs/M02/step_01.csv에 저장됩니다. 전체 결과는 같은 폴더의 DAT09_M02_result.html에서 확인합니다. 사람이 확인한 내용은 review_notes.md에 단계 번호·사례·근거·판단을 직접 적으세요. 앞 단계 자료가 없으면 이전 미션부터 순서대로 실행합니다.

### 2단계 코드와 수량 단위 및 시각 통일하기

#### 이 단계에서 하는 일

코드가 같아야 같은 대상을 찾을 수 있고, 단위와 시간대가 같아야 수량을 비교할 수 있습니다. 변환 후 값만 남기면 잘못된 환산이나 날짜 해석을 찾아내기 어렵습니다.

#### 시작 전 확인

미션2의 1단계까지 작성·확인한 결과를 사용합니다. 노트북의 m02_s2(c)를 작성하세요. c는 같은 작업의 자료와 함수를 담는 사전입니다. 필요한 결과를 c에 보관하고 화면에 보여 줄 표를 반환합니다.

#### 사용할 자료

앞 단계의 c['raw_rows'], data/catalog.json의 출처별 코드표·BOX 환산계수와 source_contracts.json의 날짜·결측 표시를 사용합니다.

#### 작업 순서

- 한 보고를 받아 복사한 뒤 공통 item_id, site_id를 붙이는 함수를 만드세요. 원래 품목·거점·수량·단위·기준시각은 raw_item, raw_site, raw_qty, raw_unit, raw_as_of로 남깁니다.

- EA는 그대로 두고 등록된 BOX는 품목의 box_to_ea를 곱해 quantity_ea를 만드세요. 이 자료의 등록 BOX는 1BOX=10EA입니다. 모르는 단위에 임의 계수를 넣지 않습니다.

- as_of·observed_at·received_at을 출처의 표기법으로 읽어 UTC로 바꾸세요. 시간대가 없거나 날짜를 해석할 수 없으면 오류로 기록하고, 서버의 현지 시간대를 추정해 붙이지 않습니다.

- 결측 여부, 숫자 여부, 단위 등록 여부, 날짜 오류를 별도로 남기세요. as_of에서 observed_at을 뺀 분 수가 관측 지연입니다. 0~60분이면 제때 보고한 것으로 표시합니다. 함수와 변환 결과를 c에 저장하세요.


#### 저장할 결과

변환 전후 값을 나란히 보여 주는 표를 반환하고 c['normalize_fn'], c['normalized']를 저장합니다. 다음 단계가 최종 normalized.csv를 저장합니다.

#### 직접 확인하는 방법

B 출처의 8BOX가 80EA가 되는지, 같은 순간을 +09:00와 UTC로 쓴 시각이 같은지 확인하세요. 공란은 숫자 0으로 바뀌면 안 됩니다.

#### 실행과 기록

이 미션의 다섯 함수를 작성한 뒤 코드셀을 실행합니다. 반환 표는 outputs/M02/step_02.csv에 저장됩니다. 전체 결과는 같은 폴더의 DAT09_M02_result.html에서 확인합니다. 사람이 확인한 내용은 review_notes.md에 단계 번호·사례·근거·판단을 직접 적으세요. 앞 단계 자료가 없으면 이전 미션부터 순서대로 실행합니다.

### 3단계 정상 수용과 보류 및 격리 구분하기

#### 이 단계에서 하는 일

수량이 없다는 사실과 잘못된 값이 들어왔다는 사실은 다릅니다. 행을 지워 버리면 무엇을 확인해야 하는지 사라집니다. 모든 보고를 남긴 채 사용할 수 있는 정도에 따라 상태와 이유를 붙입니다.

#### 시작 전 확인

미션2의 2단계까지 작성·확인한 결과를 사용합니다. 노트북의 m02_s3(c)를 작성하세요. c는 같은 작업의 자료와 함수를 담는 사전입니다. 필요한 결과를 c에 보관하고 화면에 보여 줄 표를 반환합니다.

#### 사용할 자료

c['normalized']의 변환 결과를 사용합니다. ACCEPTED는 형식 검사를 통과한 보고, HELD는 확인이 더 필요한 보고, QUARANTINED는 잘못된 값으로 격리한 보고를 뜻합니다.

#### 작업 순서

- 미등록 품목·거점, 잘못된 날짜, 숫자로 읽을 수 없는 값, 음수 또는 EA로 환산한 비정수는 QUARANTINED로 분류하세요. 여러 문제가 겹치면 이유를 모두 남깁니다.

- 앞의 격리 사유가 없는 경우 수량 결측, 미등록 단위, 기준시각보다 미래인 관측은 HELD로 둡니다. 수량 0은 결측으로 보지 않습니다.

- 그 밖의 행은 ACCEPTED로 두되 관측 지연이 60분을 넘으면 STALE 표시를 남깁니다. 형식상 수용했다는 이유만으로 최신 재고 비교에 넣지 않습니다.

- 행별 상태·이유와 상태별 건수를 저장하고, 세 상태의 합이 전체 입력과 같은지 검사하세요. 판정 함수는 c['status_fn']에 보관합니다.


#### 저장할 결과

normalized.csv, status_ledger.csv와 상태별 건수 표를 저장합니다. 오류 행을 삭제한 표로 원자료를 대체하지 않습니다.

#### 직접 확인하는 방법

기본 자료는 수용 2,135행, 결측 보류 15행, 오류 격리 10행입니다. 지연 20행은 수용 행에 포함되며 별도 네 번째 상태로 다시 더하지 않습니다.

#### 실행과 기록

이 미션의 다섯 함수를 작성한 뒤 코드셀을 실행합니다. 반환 표는 outputs/M02/step_03.csv에 저장됩니다. 전체 결과는 같은 폴더의 DAT09_M02_result.html에서 확인합니다. 사람이 확인한 내용은 review_notes.md에 단계 번호·사례·근거·판단을 직접 적으세요. 앞 단계 자료가 없으면 이전 미션부터 순서대로 실행합니다.

### 4단계 빈 수량을 채우지 않고 보완 후보 제안하기

#### 이 단계에서 하는 일

다른 두 출처가 같은 수량을 보고했다면 결측을 확인할 근거가 됩니다. 그러나 두 출처가 함께 틀렸을 수도 있으므로 실제 재고가 확정된 것은 아닙니다. 원자료와 후보를 분리해 저장합니다.

#### 시작 전 확인

미션2의 3단계까지 작성·확인한 결과를 사용합니다. 노트북의 m02_s4(c)를 작성하세요. c는 같은 작업의 자료와 함수를 담는 사전입니다. 필요한 결과를 c에 보관하고 화면에 보여 줄 표를 반환합니다.

#### 사용할 자료

c['normalized']와 미션1의 재고 이력 식별 기준을 사용합니다. 같은 대상을 찾을 때 거점·품목·지표·기준시각이 모두 같아야 합니다.

#### 작업 순서

- 이력 기준으로 세 출처를 묶고 수량이 빈 보고를 찾으세요. 각 결측 보고에 대해 나머지 두 출처의 상태·관측 지연·EA 수량을 확인합니다.

- 다른 두 출처가 모두 ACCEPTED이고 0~60분 이내 관측이며 수량이 같을 때만 그 값을 후보로 제안하세요. 둘 중 하나가 없거나 오래됐거나 수량이 다르면 NO_CANDIDATE로 둡니다.

- 대상 row_id, 거점·품목·기준시각, 원래 값, 후보값, 제안 여부와 이유, approved=False를 남기세요. 원래 공란과 상태는 바꾸지 않습니다.

- 같은 판단을 재사용할 수 있도록 후보 판정 함수를 c['missing_candidate_fn']에, 후보 목록을 c['missing_candidates']에 저장하세요.


#### 저장할 결과

candidate_missing.csv에 결측 15건의 후보와 근거를 저장합니다. 이 단계에서는 승인 마스터에 쓰지 않습니다.

#### 직접 확인하는 방법

기본 자료의 결측 15건에는 후보를 만들 수 있습니다. 다른 두 값이 10과 20이면 후보가 없어야 하고, 둘 다 최신 10이면 10을 제안할 수 있습니다.

#### 실행과 기록

이 미션의 다섯 함수를 작성한 뒤 코드셀을 실행합니다. 반환 표는 outputs/M02/step_04.csv에 저장됩니다. 전체 결과는 같은 폴더의 DAT09_M02_result.html에서 확인합니다. 사람이 확인한 내용은 review_notes.md에 단계 번호·사례·근거·판단을 직접 적으세요. 앞 단계 자료가 없으면 이전 미션부터 순서대로 실행합니다.

### 5단계 항목 변경과 경계값에 대한 시험 만들기

#### 이 단계에서 하는 일

기본 파일 한 번을 읽었다고 다음 달의 파일도 읽을 수 있다는 뜻은 아닙니다. 허용해도 되는 변화와 반드시 멈춰야 하는 변화를 구별해 시험합니다. 시험에는 예상 결과와 실제 결과가 모두 있어야 합니다.

#### 시작 전 확인

미션2의 4단계까지 작성·확인한 결과를 사용합니다. 노트북의 m02_s5(c)를 작성하세요. c는 같은 작업의 자료와 함수를 담는 사전입니다. 필요한 결과를 c에 보관하고 화면에 보여 줄 표를 반환합니다.

#### 사용할 자료

미션2의 읽기·변환·상태·후보 함수를 사용합니다. 원파일을 수정하지 말고 임시 폴더에 작은 복사 표본을 만듭니다.

#### 작업 순서

- 같은 뜻임을 확인해 등록한 stock_amount 항목과 항목 순서만 뒤집은 파일을 각각 만드세요. 두 경우는 정상으로 읽혀야 합니다.

- 재고와 뜻이 다른 출고수량 항목과 보고가 없는 빈 파일은 읽기 오류로 처리하세요. 미등록 CRATE 단위는 HELD, 수량 0은 ACCEPTED와 0EA로 확인합니다.

- 후보 판정에는 다른 두 값이 다른 경우와 한쪽 관측이 오래된 경우를 넣어 모두 후보가 없는지 확인하세요. 같은 최신값 두 개에서는 후보가 생겨야 합니다.

- 사례명·바꾼 값·예상 상태·실제 상태·일치 여부를 기록하고, 모든 시험을 자동으로 다시 실행할 수 있게 작성하세요.


#### 저장할 결과

test_results.json과 최소 여덟 사례의 결과 표를 저장합니다. 표본 파일이 임시 파일이면 만드는 코드를 남깁니다.

#### 직접 확인하는 방법

정상으로 받아야 할 두 항목 변경과 0 수량을 잘못 막지 않는지, 오류·보류 사례를 정상 수량으로 넘기지 않는지 확인하세요.

#### 실행과 기록

이 미션의 다섯 함수를 작성한 뒤 코드셀을 실행합니다. 반환 표는 outputs/M02/step_05.csv에 저장됩니다. 전체 결과는 같은 폴더의 DAT09_M02_result.html에서 확인합니다. 사람이 확인한 내용은 review_notes.md에 단계 번호·사례·근거·판단을 직접 적으세요. 앞 단계 자료가 없으면 이전 미션부터 순서대로 실행합니다.



In [ ]:
if c is None:
    raise RuntimeError("첫 셀에서 DAT09_3.md(또는 DAT09.md)를 먼저 연결하세요.")

# STEP 01 세 형식의 파일을 읽고 출처 남기기
def m02_s1(c):
    """세 형식의 파일을 읽고 출처 남기기."""
    root = c['root']
    out = folder(c, 2)
    contract = c['contract']
    required = {'row_id', 'source', 'item_code', 'site_code', 'quantity', 'unit', 'as_of', 'observed_at', 'received_at', 'metric', 'version'}

    def load_fn(source, path, extra_field_mapping=None):
        path = Path(path)
        spec = contract[source]
        fmt = spec['format']
        fm = dict(spec['field_mapping'])
        if extra_field_mapping:
            fm.update(extra_field_mapping)
        if fmt == 'csv':
            raw_list = csv_rows(path, delimiter=',')
        elif fmt == 'tsv':
            raw_list = csv_rows(path, delimiter='\t')
        elif fmt == 'json.records':
            obj = read_json(path)
            raw_list = obj['records']
        else:
            raise ValueError(f'지원하지 않는 형식: {fmt}')
        if not raw_list:
            raise ValueError(f'보고 행이 없는 빈 파일입니다: {path}')
        parsed = []
        for idx, rec in enumerate(raw_list, 1):
            mapped = {}
            for k, v in rec.items():
                if k == '출고수량':
                    raise ValueError('재고와 뜻이 다른 출고수량 항목은 읽을 수 없습니다.')
                mapped[fm.get(k, k)] = v
            missing_cols = required - set(mapped.keys())
            if missing_cols:
                raise ValueError(f'필수 항목 누락: {sorted(missing_cols)}')
            mapped['source_file'] = path.name
            mapped['line_number'] = idx
            mapped['raw_payload'] = json.dumps(clean(rec), ensure_ascii=False, sort_keys=True)
            parsed.append(mapped)
        return parsed

    sources = [
        ('A', root / 'data/supply.csv'),
        ('B', root / 'data/maintenance.json'),
        ('C', root / 'data/ammunition.tsv'),
    ]
    raw_rows = []
    manifest = {'files': [], 'total_rows': 0}
    for src, p in sources:
        rows = load_fn(src, p)
        raw_rows.extend(rows)
        manifest['files'].append({'source': src, 'file': str(p.relative_to(root)), 'row_count': len(rows), 'sha256': sha(p)})
    manifest['total_rows'] = len(raw_rows)
    write_json(out / 'ingest_manifest.json', manifest)
    csv_out(out / 'raw_rows.csv', raw_rows)
    c['load_fn'] = load_fn
    c['raw_rows'] = raw_rows
    return manifest['files']

# STEP 02 코드와 수량 단위 및 시각 통일하기
def m02_s2(c):
    """코드와 수량 단위 및 시각 통일하기."""
    cat = c['catalog']
    contract = c['contract']
    item_meta = {x['item_id']: x for x in cat['items']}

    def normalize_fn(row):
        nr = dict(row)
        src = nr.get('source', '')
        nr['raw_item'] = row.get('item_code')
        nr['raw_site'] = row.get('site_code')
        nr['raw_qty'] = row.get('quantity')
        nr['raw_unit'] = row.get('unit')
        nr['raw_as_of'] = row.get('as_of')

        item_map = cat['mapping'].get(src, {}).get('item', {})
        site_map = cat['mapping'].get(src, {}).get('site', {})
        nr['item_id'] = item_map.get(nr['raw_item'])
        nr['site_id'] = site_map.get(nr['raw_site'])

        tokens = set(contract.get(src, {}).get('missing_tokens', ['', 'NULL', 'N/A']))
        q_val = nr['raw_qty']
        is_missing = (q_val is None) or (isinstance(q_val, str) and q_val.strip() in tokens)
        nr['is_missing'] = is_missing

        q_num = None
        is_numeric = False
        if not is_missing:
            try:
                q_num = float(q_val)
                if math.isfinite(q_num):
                    is_numeric = True
                else:
                    q_num = None
            except Exception:
                is_numeric = False
        nr['is_numeric'] = is_numeric

        unit = nr['raw_unit']
        unit_reg = unit in ('EA', 'BOX')
        nr['unit_registered'] = unit_reg
        qty_ea = None
        is_integral_ea = False
        if is_numeric and unit_reg:
            factor = 1 if unit == 'EA' else item_meta.get(nr['item_id'], {}).get('box_to_ea', 10)
            conv = q_num * factor
            if float(conv).is_integer():
                qty_ea = int(round(conv))
                is_integral_ea = True
            else:
                qty_ea = conv
        nr['quantity_ea'] = qty_ea
        nr['is_integral_ea'] = is_integral_ea

        date_err = False
        lag_min = None
        for col in ('as_of', 'observed_at', 'received_at'):
            try:
                dt_utc = parse_time(nr.get(col), src)
                nr[col] = dt_utc.isoformat()
            except Exception:
                date_err = True
        if not date_err:
            t_as = parse_time(nr['as_of'], 'A')
            t_obs = parse_time(nr['observed_at'], 'A')
            lag_min = (t_as - t_obs).total_seconds() / 60.0
        nr['date_error'] = date_err
        nr['lag_minutes'] = lag_min
        nr['is_timely'] = (not date_err) and (lag_min is not None) and (0 <= lag_min <= 60)
        try:
            nr['version'] = int(nr.get('version'))
        except Exception:
            nr['version'] = None
        return nr

    normalized = [normalize_fn(r) for r in c['raw_rows']]
    c['normalize_fn'] = normalize_fn
    c['normalized'] = normalized
    return [
        {
            'row_id': r['row_id'], 'source': r['source'],
            'raw_item': r['raw_item'], 'item_id': r['item_id'],
            'raw_site': r['raw_site'], 'site_id': r['site_id'],
            'raw_qty': r['raw_qty'], 'raw_unit': r['raw_unit'], 'quantity_ea': r['quantity_ea'],
            'raw_as_of': r['raw_as_of'], 'as_of': r['as_of'],
            'lag_minutes': r['lag_minutes'], 'is_missing': r['is_missing'], 'date_error': r['date_error'],
        }
        for r in normalized
    ]

# STEP 03 정상 수용과 보류 및 격리 구분하기
def m02_s3(c):
    """정상 수용과 보류 및 격리 구분하기."""
    out = folder(c, 2)

    def status_fn(nr):
        q_reasons = []
        if not nr.get('item_id'):
            q_reasons.append('UNREGISTERED_ITEM')
        if not nr.get('site_id'):
            q_reasons.append('UNREGISTERED_SITE')
        if nr.get('metric') != 'on_hand':
            q_reasons.append('INVALID_METRIC')
        if nr.get('date_error'):
            q_reasons.append('DATE_ERROR')
        if not nr.get('is_missing') and not nr.get('is_numeric'):
            q_reasons.append('NON_NUMERIC_QTY')
        if nr.get('is_numeric'):
            if float(nr['raw_qty']) < 0:
                q_reasons.append('NEGATIVE_QTY')
            elif nr.get('unit_registered') and not nr.get('is_integral_ea'):
                q_reasons.append('NON_INTEGRAL_EA')
        if nr.get('version') is None or nr['version'] <= 0:
            q_reasons.append('INVALID_VERSION')
        if q_reasons:
            return 'QUARANTINED', q_reasons, False

        h_reasons = []
        if nr.get('is_missing'):
            h_reasons.append('MISSING_QTY')
        if not nr.get('unit_registered'):
            h_reasons.append('UNREGISTERED_UNIT')
        if nr.get('lag_minutes') is not None and nr['lag_minutes'] < 0:
            h_reasons.append('FUTURE_OBSERVATION')
        if h_reasons:
            return 'HELD', h_reasons, False

        stale = (nr.get('lag_minutes') is not None) and (nr['lag_minutes'] > 60)
        reasons = ['STALE'] if stale else ['OK']
        return 'ACCEPTED', reasons, stale

    ledger = []
    counts = {'ACCEPTED': 0, 'HELD': 0, 'QUARANTINED': 0, 'STALE': 0}
    for nr in c['normalized']:
        st, reasons, stale = status_fn(nr)
        nr['status'] = st
        nr['reasons'] = '|'.join(reasons)
        nr['stale_flag'] = stale
        counts[st] += 1
        if stale:
            counts['STALE'] += 1
        ledger.append({
            'row_id': nr['row_id'], 'source': nr['source'],
            'item_id': nr['item_id'], 'site_id': nr['site_id'],
            'as_of': nr['as_of'], 'quantity_ea': nr['quantity_ea'],
            'status': st, 'reasons': nr['reasons'], 'stale_flag': stale,
        })
    assert sum(counts[k] for k in ('ACCEPTED', 'HELD', 'QUARANTINED')) == len(c['normalized'])
    csv_out(out / 'normalized.csv', c['normalized'])
    csv_out(out / 'status_ledger.csv', ledger)
    c['status_fn'] = status_fn
    c['status_counts'] = counts
    return ledger

# STEP 04 빈 수량을 채우지 않고 보완 후보 제안하기
def m02_s4(c):
    """빈 수량을 채우지 않고 보완 후보 제안하기."""
    out = folder(c, 2)

    def missing_candidate_fn(target_row, peer_rows):
        timely_peers = [
            p for p in peer_rows
            if p['row_id'] != target_row['row_id']
            and p.get('status') == 'ACCEPTED'
            and not p.get('stale_flag')
            and p.get('lag_minutes') is not None and 0 <= p['lag_minutes'] <= 60
            and p.get('quantity_ea') is not None
        ]
        if len(timely_peers) >= 2:
            vals = {p['quantity_ea'] for p in timely_peers}
            if len(vals) == 1:
                cand_val = next(iter(vals))
                return {
                    'row_id': target_row['row_id'],
                    'site_id': target_row['site_id'],
                    'item_id': target_row['item_id'],
                    'as_of': target_row['as_of'],
                    'raw_qty': target_row['raw_qty'],
                    'candidate_qty_ea': cand_val,
                    'proposal_status': 'PROPOSED',
                    'reason': f"다른 두 정상·적시 출처 일치({', '.join(p['row_id'] for p in timely_peers)})",
                    'approved': False,
                }
        return {
            'row_id': target_row['row_id'],
            'site_id': target_row['site_id'],
            'item_id': target_row['item_id'],
            'as_of': target_row['as_of'],
            'raw_qty': target_row['raw_qty'],
            'candidate_qty_ea': None,
            'proposal_status': 'NO_CANDIDATE',
            'reason': '정상·적시 두 출처 일치 조건 미충족',
            'approved': False,
        }

    by_key = {}
    for nr in c['normalized']:
        if nr.get('site_id') and nr.get('item_id') and not nr.get('date_error'):
            by_key.setdefault(logical_key(nr), []).append(nr)

    missing_cands = []
    for nr in c['normalized']:
        if nr.get('is_missing'):
            peers = by_key.get(logical_key(nr), [])
            missing_cands.append(missing_candidate_fn(nr, peers))
    csv_out(out / 'candidate_missing.csv', missing_cands)
    c['missing_candidate_fn'] = missing_candidate_fn
    c['missing_candidates'] = missing_cands
    return missing_cands

# STEP 05 항목 변경과 경계값에 대한 시험 만들기
def m02_s5(c):
    """항목 변경과 경계값에 대한 시험 만들기."""
    out = folder(c, 2)
    load_fn = c['load_fn']
    norm_fn = c['normalize_fn']
    stat_fn = c['status_fn']
    cand_fn = c['missing_candidate_fn']
    tests = []

    with tempfile.TemporaryDirectory() as tmp:
        tp = Path(tmp)
        # 1. stock_amount 별칭 허용
        p1 = tp / 'stock_amount.csv'
        p1.write_text('row_id,source,material,depot,metric,stock_amount,unit,report_at,observed_at,received_at,version\nT-01,A,S001,DEP001,on_hand,50,EA,2026-10-01T09:00:00+09:00,2026-10-01T08:55:00+09:00,2026-10-01T09:01:00+09:00,1\n', encoding='utf-8')
        r1 = load_fn('A', p1, extra_field_mapping={'stock_amount': 'quantity'})[0]
        tests.append({'case_id': 'T01_stock_amount_alias', 'changed_value': 'stock->stock_amount', 'expected': 'LOAD_OK', 'actual': 'LOAD_OK' if r1['quantity'] == '50' else 'FAIL', 'passed': r1['quantity'] == '50'})

        # 2. 열 순서 뒤집기 허용
        p2 = tp / 'reversed_cols.csv'
        p2.write_text('version,received_at,observed_at,report_at,unit,stock,metric,depot,material,source,row_id\n1,2026-10-01T09:01:00+09:00,2026-10-01T08:55:00+09:00,2026-10-01T09:00:00+09:00,EA,50,on_hand,DEP001,S001,A,T-02\n', encoding='utf-8')
        r2 = load_fn('A', p2)[0]
        tests.append({'case_id': 'T02_reversed_columns', 'changed_value': 'columns reversed', 'expected': 'LOAD_OK', 'actual': 'LOAD_OK' if r2['quantity'] == '50' else 'FAIL', 'passed': r2['quantity'] == '50'})

        # 3. 출고수량 거부
        p3 = tp / 'issue_qty.csv'
        p3.write_text('row_id,source,material,depot,metric,출고수량,unit,report_at,observed_at,received_at,version\nT-03,A,S001,DEP001,on_hand,10,EA,2026-10-01T09:00:00+09:00,2026-10-01T08:55:00+09:00,2026-10-01T09:01:00+09:00,1\n', encoding='utf-8')
        try:
            load_fn('A', p3)
            act3 = 'LOAD_OK'
        except ValueError:
            act3 = 'LOAD_ERROR'
        tests.append({'case_id': 'T03_reject_issue_qty', 'changed_value': 'stock->출고수량', 'expected': 'LOAD_ERROR', 'actual': act3, 'passed': act3 == 'LOAD_ERROR'})

        # 4. 빈 파일 거부
        p4 = tp / 'empty.csv'
        p4.write_text('row_id,source,material,depot,metric,stock,unit,report_at,observed_at,received_at,version\n', encoding='utf-8')
        try:
            load_fn('A', p4)
            act4 = 'LOAD_OK'
        except ValueError:
            act4 = 'LOAD_ERROR'
        tests.append({'case_id': 'T04_reject_empty_file', 'changed_value': '0 rows', 'expected': 'LOAD_ERROR', 'actual': act4, 'passed': act4 == 'LOAD_ERROR'})

    # 5. CRATE 미등록 단위 -> HELD
    nr5 = norm_fn({'row_id': 'T-05', 'source': 'A', 'item_code': 'S001', 'site_code': 'DEP001', 'metric': 'on_hand', 'quantity': '5', 'unit': 'CRATE', 'as_of': '2026-10-01T09:00:00+09:00', 'observed_at': '2026-10-01T08:55:00+09:00', 'received_at': '2026-10-01T09:01:00+09:00', 'version': '1'})
    st5, _, _ = stat_fn(nr5)
    tests.append({'case_id': 'T05_unregistered_unit_CRATE', 'changed_value': 'unit=CRATE', 'expected': 'HELD', 'actual': st5, 'passed': st5 == 'HELD' and nr5['quantity_ea'] is None})

    # 6. 수량 0 -> ACCEPTED & 0EA
    nr6 = norm_fn({'row_id': 'T-06', 'source': 'A', 'item_code': 'S001', 'site_code': 'DEP001', 'metric': 'on_hand', 'quantity': '0', 'unit': 'EA', 'as_of': '2026-10-01T09:00:00+09:00', 'observed_at': '2026-10-01T08:55:00+09:00', 'received_at': '2026-10-01T09:01:00+09:00', 'version': '1'})
    st6, _, _ = stat_fn(nr6)
    tests.append({'case_id': 'T06_zero_quantity', 'changed_value': 'quantity=0', 'expected': 'ACCEPTED_0EA', 'actual': f"{st6}_{nr6['quantity_ea']}EA", 'passed': st6 == 'ACCEPTED' and nr6['quantity_ea'] == 0})

    # 7. 두 정상 출처 불일치(10 vs 20) -> NO_CANDIDATE
    base_t = {'row_id': 'X0', 'site_id': 'U001', 'item_id': 'I001', 'as_of': '2026-10-01T00:00:00+00:00', 'raw_qty': ''}
    c7 = cand_fn(base_t, [
        {'row_id': 'X1', 'status': 'ACCEPTED', 'stale_flag': False, 'lag_minutes': 5, 'quantity_ea': 10},
        {'row_id': 'X2', 'status': 'ACCEPTED', 'stale_flag': False, 'lag_minutes': 5, 'quantity_ea': 20},
    ])
    tests.append({'case_id': 'T07_candidate_mismatch', 'changed_value': 'peers=10,20', 'expected': 'NO_CANDIDATE', 'actual': c7['proposal_status'], 'passed': c7['proposal_status'] == 'NO_CANDIDATE'})

    # 8. 한쪽 지연(125분) -> NO_CANDIDATE
    c8 = cand_fn(base_t, [
        {'row_id': 'X1', 'status': 'ACCEPTED', 'stale_flag': False, 'lag_minutes': 5, 'quantity_ea': 10},
        {'row_id': 'X2', 'status': 'ACCEPTED', 'stale_flag': True, 'lag_minutes': 125, 'quantity_ea': 10},
    ])
    tests.append({'case_id': 'T08_candidate_stale_peer', 'changed_value': 'lag=5,125', 'expected': 'NO_CANDIDATE', 'actual': c8['proposal_status'], 'passed': c8['proposal_status'] == 'NO_CANDIDATE'})

    # 9. 두 정상·적시 출처 일치(10 vs 10) -> PROPOSED
    c9 = cand_fn(base_t, [
        {'row_id': 'X1', 'status': 'ACCEPTED', 'stale_flag': False, 'lag_minutes': 5, 'quantity_ea': 10},
        {'row_id': 'X2', 'status': 'ACCEPTED', 'stale_flag': False, 'lag_minutes': 10, 'quantity_ea': 10},
    ])
    tests.append({'case_id': 'T09_candidate_match', 'changed_value': 'peers=10,10', 'expected': 'PROPOSED_10', 'actual': f"{c9['proposal_status']}_{c9['candidate_qty_ea']}", 'passed': c9['proposal_status'] == 'PROPOSED' and c9['candidate_qty_ea'] == 10})

    write_json(out / 'test_results.json', {'total': len(tests), 'passed': sum(t['passed'] for t in tests), 'cases': tests})
    (out / 'review_notes.md').write_text(
        '# M02 검토 기록\n- 전체 2,160행 적재 및 변환 완료(ACCEPTED 2,135 / HELD 15 / QUARANTINED 10 / STALE 20).\n'
        '- B-0000(8BOX -> 80EA) 등 BOX=10EA 환산, 공란 15건 결측 보존 및 후보 분리 확인.\n'
        '- 9개 단위·스키마 시험 모두 통과.\n',
        encoding='utf-8',
    )
    return tests

result_html = run_mission(c, 2, [m02_s1, m02_s2, m02_s3, m02_s4, m02_s5])
document = result_html.read_text(encoding="utf-8")
display(HTML('<iframe title="DAT09 단계별 결과" style="width:100%;height:1100px;border:1px solid #ccd5dd" sandbox="allow-scripts allow-downloads allow-same-origin" srcdoc="'+html.escape(document,quote=True)+'"></iframe>'))
